# Mechanism transfer: predicting an UNSEEN ligand class

**The question.** Two notions of receptor similarity compete. Raw **ESM** carries mediocre
*structural* (sequence) information. Our **graph-refined** embedding replaces part of that
structure with *function* learned from binding. Which one better encodes a **mechanism** —
tested on a chemical class the graph **never saw during training**?

**Weak (testable) formulation.** Pick a chemically-coherent, mechanistically-meaningful
odorant class C (carboxylic acids -> the Class I "acid receptor" mechanism; thiols -> the
Cu-coordination mechanism). **Remove every molecule of C from training entirely.** Train the
receptor graph on the rest, so its receptor embeddings are built without a single example of
C. Then ask: does the refined receptor geometry predict each receptor's **held-out response
to C** better than raw ESM? If yes, the embedding encodes a *transferable mechanism*, not
memorised statistics — the mechanism "shows itself" on unseen examples.

We test our **titular** pipeline (q99 signed graph) **and** a **full-coverage** variant
(q=0, no quantile), over **5 seeds**, with a GPU attached in-notebook.

*Honest caveat.* A broadly-tuned receptor binds a bit of everything, so some transfer is
generic tuning rather than the specific C-mechanism. Raw ESM (which sees no binding at all)
is the control: any refined-over-ESM gain is binding-derived signal that generalised to the
unseen class. M2OR is large and diverse, so it cannot be fully "surprised" — read the gaps,
not the absolute numbers.


In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, t as _t

from orbind.regimes import full_full_pairs
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict, _train_one, MOL, PROT)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# ---- configuration -------------------------------------------------------
PROT_NPZ = DATA / "embeddings/proteins/esm1b_650m_mean.npz"
MOL_NPZ  = DATA / "embeddings/molecules/gin_supervised_contextpred_all_m2or.npz"

# graph-config variants: our titular q99 signed graph, and full-coverage (q=0, no quantile)
VARIANTS = {
    "q99 (titular)": dict(q=0.99, criterion="greedy_pair_cover", k_mode="coverage_quantile"),
    "full (q=0)":    dict(q=0.0,  criterion="coverage",          k_mode="coverage_quantile"),
}
SEEDS = [42, 43, 44, 45, 46]
EPOCHS = 900          # titular protocol; lower for a quick dry run

# mechanistically-meaningful odorant classes (SMARTS on the SMILES)
SMARTS = {
    "carboxylic_acid": "[CX3](=O)[OX2H1]",   # Class I acid-receptor mechanism
    "thiol":           "[#16X2H]",           # Cu/Zn-coordination mechanism
    "aldehyde":        "[CX3H1](=O)[#6]",
    "ester":           "[#6][CX3](=O)[OX2H0][#6]",
}
RUN_CLASSES = ["carboxylic_acid", "thiol"]   # add more here to widen the sweep
K_NN = 10                                    # neighbours for the LOO readout

## 1 · Molecules per class (RDKit) and the held-out receptor labels

Each class is a SMARTS substructure. A receptor's **held-out label** is its response to that
class: `1` if it binds at least one class-C molecule (among those it was tested on), else `0`
— computed only from the C sub-matrix, which never enters training.


In [ ]:
pairs = full_full_pairs()
uniq  = pairs.drop_duplicates("inchikey")[["inchikey", "smiles"]]
mols  = {ik: Chem.MolFromSmiles(s) for ik, s in zip(uniq["inchikey"], uniq["smiles"])}

def class_molecules(cname):
    q = Chem.MolFromSmarts(SMARTS[cname])
    return {ik for ik, m in mols.items() if m is not None and m.HasSubstructMatch(q)}

CLASS_IKS = {c: class_molecules(c) for c in RUN_CLASSES}
for c, iks in CLASS_IKS.items():
    sub = pairs[pairs["inchikey"].isin(iks)]
    resp = sub[sub["label"] == 1]["receptor"].nunique()
    print(f"{c:16} molecules={len(iks):3d} | receptors tested-on={sub['receptor'].nunique():4d} "
          f"| responders(>=1)={resp:4d}")

## 2 · Train the graph WITHOUT the class, per variant x seed

For each class we drop all its molecules, build the universe on what remains, and train the
signed graph (titular and full-coverage), five seeds each. `emit="prot"` returns the refined
receptor vector — trained without a single class-C example. Raw ESM is the same for all and
serves as the structural baseline.


In [ ]:
def refined_receptors(variant_kw, seed, drop_iks):
    """Train the signed graph on all pairs whose molecule is NOT in drop_iks;
    return {receptor: refined vector} and {receptor: raw ESM vector}."""
    ext = GnnSignedExtractor(name="cls", protein_path=str(PROT_NPZ), molecule_path=str(MOL_NPZ),
                             emit="prot", n_models=1, hidden=256, edge_threshold=0.0,
                             task="classification", epochs=EPOCHS, **variant_kw)
    keep = ~pairs["inchikey"].isin(drop_iks)
    idx  = np.where(keep.to_numpy())[0]
    idx  = idx[ext.covered(pairs, idx)]
    mol_to_i, prot_to_i, x_mol, x_prot = _build_universe(pairs, idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(pairs, idx, mol_to_i, prot_to_i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pe, ne = _edge_index_dict(pos, neg)
    sub = pairs.iloc[idx]
    mi = sub["inchikey"].map(mol_to_i).to_numpy()
    pi = sub["receptor"].map(prot_to_i).to_numpy()
    y  = sub["label"].to_numpy(dtype=np.float32)
    torch.manual_seed(seed)
    _, z_prot, _ = _train_one(ext._build_model, x_mol, x_prot, pe, ne, mi, pi, y,
                              ext._hp(seed), DEVICE, checkpoint_path=None)
    refined = {r: z_prot[i] for r, i in prot_to_i.items()}
    rawesm  = {r: x_prot[i].numpy() for r, i in prot_to_i.items()}
    return refined, rawesm

In [ ]:
def heldout_label(cname, receptors):
    """Binary C-response + the per-C-molecule profile, for the given receptors."""
    iks = CLASS_IKS[cname]
    sub = pairs[pairs["inchikey"].isin(iks)]
    cmols = sorted(iks)
    m_idx = {m: j for j, m in enumerate(cmols)}
    prof  = {}
    label = {}
    for r, g in sub.groupby("receptor"):
        v = np.full(len(cmols), np.nan)
        agg = g.groupby("inchikey")["label"].max()
        for ik, val in agg.items():
            v[m_idx[ik]] = val
        prof[r]  = v
        label[r] = int(np.nanmax(v) >= 1) if np.isfinite(v).any() else None
    recs = [r for r in receptors if label.get(r) is not None]
    y    = np.array([label[r] for r in recs])
    Pc   = np.stack([np.nan_to_num(prof[r], nan=0.0) for r in recs])  # signed-ish profile (0/1)
    return recs, y, Pc

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def loo_knn_auroc(X, y, k=K_NN):
    S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    score = np.array([y[row].mean() for row in nn])
    return roc_auc_score(y, score) if 0 < y.sum() < len(y) else np.nan

def profile_spearman(X, Pc):
    Fc = Pc - Pc.mean(1, keepdims=True)
    Fn = Fc / (np.linalg.norm(Fc, axis=1, keepdims=True) + 1e-9)
    F  = Fn @ Fn.T
    iu = np.triu_indices(len(X), 1)
    return spearmanr(F[iu], emb_sim(X)[iu]).correlation

In [ ]:
rows = []
for cname in RUN_CLASSES:
    drop = CLASS_IKS[cname]
    # raw ESM baseline is variant/seed-independent: take it from any trained universe once
    base_ref, base_raw = refined_receptors(VARIANTS[next(iter(VARIANTS))], SEEDS[0], drop)
    recs, y, Pc = heldout_label(cname, list(base_raw))
    Xesm = np.stack([base_raw[r] for r in recs])
    rows.append(dict(cls=cname, embedding="raw ESM", variant="-", seed="-",
                     auroc=loo_knn_auroc(Xesm, y), spear=profile_spearman(Xesm, Pc)))
    print(f"[{cname}] {len(recs)} receptors, {int(y.sum())} responders | "
          f"raw ESM AUROC {rows[-1]['auroc']:.3f}")
    for vname, vkw in VARIANTS.items():
        for seed in SEEDS:
            ref, _ = refined_receptors(vkw, seed, drop)
            Xr = np.stack([ref[r] for r in recs])
            rows.append(dict(cls=cname, embedding=f"refined {vname}", variant=vname, seed=seed,
                             auroc=loo_knn_auroc(Xr, y), spear=profile_spearman(Xr, Pc)))
        au = [r["auroc"] for r in rows if r["cls"] == cname and r["variant"] == vname]
        print(f"    refined {vname:14} AUROC {np.nanmean(au):.3f} +/- {np.nanstd(au):.3f}")
res = pd.DataFrame(rows)

## 3 · Who encodes the mechanism? ESM vs refined, on the unseen class

AUROC of predicting the held-out class response from the receptor geometry (leave-one-out
kNN). Raw ESM = structure only; refined (both graph variants) = structure partly replaced by
function. A refined bar above the ESM line means the graph transferred a mechanism it never
trained on.


In [ ]:
def ci(x):
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    if len(x) < 2: return (np.nanmean(x) if len(x) else np.nan, np.nan)
    return x.mean(), _t.ppf(.975, len(x) - 1) * x.std(ddof=1) / len(x) ** .5

summary = []
for cname in RUN_CLASSES:
    esm = res[(res.cls == cname) & (res.embedding == "raw ESM")]["auroc"].iloc[0]
    row = {"class": cname, "raw ESM": esm}
    for vname in VARIANTS:
        m, h = ci(res[(res.cls == cname) & (res.variant == vname)]["auroc"])
        row[f"refined {vname}"] = m
        row[f"refined {vname} _ci"] = h
    summary.append(row)
summ = pd.DataFrame(summary)
display(summ[[c for c in summ.columns if not c.endswith("_ci")]]
        .style.format({c: "{:.3f}" for c in summ.columns if c not in ("class",) and not c.endswith("_ci")})
        .hide(axis="index"))

# tuning baseline: how well does a receptor's OVERALL non-C hit-rate alone predict its
# class response? Any AUROC near this is generic tuning, not the class mechanism.
tune_auroc = {}
for cname in RUN_CLASSES:
    hit = pairs[~pairs["inchikey"].isin(CLASS_IKS[cname])].groupby("receptor")["label"].mean()
    recs_c, y_c, _ = heldout_label(cname, list(hit.index))
    hr = hit.loc[recs_c].to_numpy()
    tune_auroc[cname] = roc_auc_score(y_c, hr) if 0 < y_c.sum() < len(y_c) else np.nan
    print(f"{cname:16} tuning-only baseline AUROC = {tune_auroc[cname]:.3f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
labels = ["raw ESM"] + [f"refined {v}" for v in VARIANTS]
colors = {"raw ESM": "#888888", labels[1]: "#4c78a8", labels[2]: "#e4794c"}
x = np.arange(len(RUN_CLASSES)); w = 0.26
for j, lab in enumerate(labels):
    vals = [summ.loc[summ["class"] == c, lab].iloc[0] for c in RUN_CLASSES]
    errs = [0 if lab == "raw ESM" else summ.loc[summ["class"] == c, f"{lab} _ci"].iloc[0] for c in RUN_CLASSES]
    ax.bar(x + (j - 1) * w, vals, w, yerr=errs, capsize=3, label=lab, color=colors[lab])
for xi, c in enumerate(RUN_CLASSES):
    ax.plot([xi - 0.45, xi + 0.45], [tune_auroc[c]] * 2, ls="--", c="crimson", lw=1.3,
            label="tuning-only baseline" if xi == 0 else None)
ax.axhline(0.5, ls=":", c="gray")
ax.set_xticks(x); ax.set_xticklabels(RUN_CLASSES)
ax.set_ylabel(f"held-out class AUROC (LOO {K_NN}-NN)")
ax.set_title("Predicting an UNSEEN ligand class: structure (ESM) vs function-refined")
ax.legend(frameon=False); ax.set_ylim(0.4, 1.0); fig.tight_layout(); plt.show()

**Reading it.** If the refined bars clear the raw-ESM baseline on a class the graph never
trained on, the receptor embedding carries a **transferable binding mechanism**, not just
memorised co-occurrence — the strongest sense in which our receptor side beats ESM. If they
sit at or below ESM, the honest conclusion is that on a dataset as broad as M2OR the graph
mostly re-encodes statistics it already had. Comparing the **titular q99** and **full-coverage
q0** variants shows whether the hub-molecule restriction helps or hurts this transfer.

**Two caveats are drawn on the plot.** M2OR is sparse, so *which* receptor x molecule pairs were
assayed is not random — a confound this binary readout cannot fully remove; and the **red dashed
tuning-only baseline** is the AUROC obtained from each receptor's overall non-C hit-rate alone
(generic responsiveness). Any bar near that line is tuning, not the class mechanism — only the
margin *above* it is a specific-mechanism signal. The insect complete-matrix notebook
(`mechanism_holdout_insects.ipynb`) removes both confounds by construction (uniform assay
pattern) and by residualising the target against tuning; treat this M2OR result as the
illustrative version and that one as the clean test.
